
# Construção do `gerador.cpp` e `sorts.cpp`

Este notebook descreve a construção dos dois principais programas do projeto:

1. `gerador.cpp`
2. `sorts.cpp`

O objetivo é documentar:
- estrutura dos programas;
- metodologia experimental;
- geração dos datasets;
- execução dos benchmarks.

Foi utilizado o agente de IA do ChatGPT para ajudar a fazer ambos programas.



# Objetivo Geral do Projeto

O projeto busca comparar algoritmos de ordenação utilizando diferentes padrões de acesso à memória e diferentes estruturas de entrada.

Os algoritmos avaliados são:

- HeapSort
- MergeSort
- QuickSort



# Construção do `gerador.cpp`

O `gerador.cpp` é responsável por:

1. Criar os datasets;
2. Garantir reprodutibilidade;
3. Salvar os vetores em formato binário.

O programa permite seeds de entrada para possibilitar a reprodutibilidade. Dando default para seed 1 caso nenhum seed de entrada seja informada.



# Estrutura dos Datasets

O gerador cria 8 vetores de tamanho definido pela entrada de usuário:

- ordered
- reverse
- random
- repeated
- repeated_1_5
- zigzag
- noise_end
- random_swaps

Cada dataset foi projetado para provocar diferentes comportamentos dos algoritmos.



# Formato Binário

Os vetores são armazenados em formato `.bin`.

Isso foi escolhido porque:
- leitura é mais rápida;
- arquivos ocupam menos espaço;
- reduz overhead experimental.



# Construção do `sorts.cpp`

O `sorts.cpp` realiza:

1. Leitura dos datasets;
2. Aplicação ou não de estress;
3. Execução dos algoritmos;
4. Medição de métricas;
5. Salvamento em CSV.



# Entrada por Linha de Comando

O programa é executado no formato:

```bash
./sorts <algoritmo> <arquivo_bin> stress
```

Exemplo:

```bash
./sorts heap datasets/random.bin cpu
```



# Isolamento Experimental

Cada execução do benchmark realiza apenas:

- um algoritmo de ordenação;
- um conjunto de dados (dataset);
- um nível de estresse (nenhum, CPU, RAM ou ambos).

Quando o nível de estresse é aplicado, são executadas no máximo:

- uma thread de estresse de CPU;
- uma thread de estresse de RAM.

Essa estratégia proporciona:

- isolamento entre os experimentos;
- medições mais confiáveis de tempo, memória e energia;
- menor interferência entre execuções;
- maior reprodutibilidade dos resultados.



# Medição de Tempo

A medição utiliza:

```cpp
std::chrono::high_resolution_clock
```

Os tempos são armazenados em milissegundos.

# Uso de Memória RAM

A medição utiliza
```cpp
VmHWM (pico de uso RAM)
```
Também é medida RAM média

# Gasto Energético

A medição utiliza
```cpp
intel-rapl:0/energy_uj
```

Desativada temporariamente para testagem

# Comparações

A métrica de comparações contabiliza o número total de vezes em que duas entradas são avaliadas em operações condicionais do algoritmo (ex: a < b, a > b, a <= b).

Cada avaliação lógica relevante é instrumentada para incrementar um contador global de comparações.

Essa métrica permite avaliar a complexidade real do algoritmo independente do hardware.

# Swaps

A métrica de swaps contabiliza o número total de trocas de posições realizadas entre elementos do conjunto de dados.

Cada operação de troca (swap(a, b)) incrementa um contador global.

Essa métrica permite medir o custo de reorganização dos dados durante a execução do algoritmo, sendo especialmente relevante para algoritmos de ordenação como QuickSort e HeapSort.

# Estresse de CPU

Para avaliar o impacto da contenção por recursos computacionais, foi implementada uma thread independente responsável por gerar carga contínua sobre a CPU durante toda a execução do algoritmo de ordenação.

O estresse consiste na execução repetitiva de operações aritméticas de ponto flutuante (multiplicação, divisão, soma e subtração), utilizando uma variável `volatile` para impedir que o compilador elimine as operações por otimização.

A thread permanece ativa durante toda a execução do algoritmo e é encerrada imediatamente após sua conclusão.

```cpp
void runCpuStress(std::atomic<bool>& running)
{
    volatile double x = 1.1;

    while (running)
    {
        x *= 1.0000001;
        x /= 1.00000009;
        x += 0.1234;
        x -= 0.5678;
        x *= 1.0000003;
        x /= 1.0000002;
        x += x * 0.000001;
    }
}
```

Esse método mantém ocupadas as unidades de execução da CPU sem realizar operações de entrada e saída (I/O) ou criar múltiplas threads de processamento, permitindo analisar o impacto da competição pelos recursos do processador sobre o desempenho dos algoritmos.

---

# Estresse de Memória RAM

Para avaliar o impacto da competição pelo subsistema de memória, foi implementada uma thread dedicada à geração contínua de acessos à memória principal (RAM).

A thread aloca um buffer de **512 MB**, tamanho significativamente superior às caches dos processadores utilizados, reduzindo a probabilidade de reutilização dos dados armazenados em cache.

Os endereços acessados são gerados por meio do algoritmo **XorShift64**, produzindo uma sequência pseudoaleatória que dificulta a atuação dos mecanismos de prefetch do hardware.

Cada acesso realiza uma leitura seguida de uma escrita sobre o buffer, aumentando a utilização da largura de banda da memória principal. A thread permanece ativa durante toda a execução do algoritmo de ordenação e é finalizada ao término do experimento.

```cpp
void runRamStress(std::atomic<bool>& running)
{
    constexpr size_t chunkSize = 512ULL * 1024ULL * 1024ULL;
    constexpr size_t mask = chunkSize - 1;

    std::vector<unsigned char> buffer(chunkSize, 0);

    uint64_t state = 0x123456789ABCDEFULL;
    unsigned char value = 0;

    while (running)
    {
        for (int i = 0; i < 8; i++)
        {
            state ^= state << 13;
            state ^= state >> 7;
            state ^= state << 17;

            size_t index = state & mask;

            value ^= buffer[index];
            buffer[index] = value;
        }
    }
}
```

Esse método gera um fluxo contínuo de leituras e escritas em posições pseudoaleatórias da memória, aumentando a pressão sobre a largura de banda da RAM e permitindo avaliar como algoritmos com diferentes padrões de acesso à memória respondem à contenção desse recurso.

# Validação Sorting

Todo sorting antes de ser salvo é validado que foi realizado corretamente, sortings errados não são salvos nas estatísticas


# Resultados CSV

Os resultados são salvos em:

```text
results/(tamanho vetor).csv
```

Formato:

```csv
vector_type,algorithm,time_ms,stress,ram_mb,energy_j
```

Também são criados .csv únicos a tipos de vetor e tipos de algoritmo
